# 05-3. 보안 로그 정규식 적용

## Goal

웹로그·SSH·CloudTrail에서 검사할 필드와 문자열 범위를 선택한다. 형식 일치와 공격 판단을 구분한다.

기존 05-3 TODO 실습 이후 참고하는 실행 예제다. 외부 네트워크 요청은 보내지 않는다.


## Setup

05장 교육자료 ZIP을 풀고 `notebooks`와 `examples`의 구조를 유지한다. Python 표준 라이브러리만 사용한다. 입력과 패턴은 `examples/05-security-regex/security_regex.py`에 있다.


In [1]:
from pathlib import Path
import importlib.util
import json

for root in (Path.cwd(), *Path.cwd().parents):
    module_path = root / "examples/05-security-regex/security_regex.py"
    if module_path.is_file():
        break
else:
    raise FileNotFoundError("05장 ZIP의 notebooks와 examples 폴더를 함께 유지하세요.")

spec = importlib.util.spec_from_file_location("security_regex", module_path)
lesson = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lesson)
print("합성 보안 로그 예제 로드 완료")


합성 보안 로그 예제 로드 완료


## Steps

### 1. 웹 상태 코드와 SSH 실패 메시지

실행 전 URL 속 404가 후보로 선택될지 예상한다. 웹 형식 미지원도 불일치하므로 이 함수로 정상 여부를 판단하지 않는다.


In [2]:
result = lesson.summarize()
print("웹 상태 후보 행:", result["web_error_lines"])
print("SSH 실패 행:", result["ssh_failure_lines"])
assert result["web_error_lines"] == [1, 2]
assert result["ssh_failure_lines"] == [1, 2]


웹 상태 후보 행: [1, 2]
SSH 실패 행: [1, 2]


### 2. CloudTrail 필드별 검사

JSON을 먼저 읽고 eventName·errorCode·userAgent를 각각 검사한다. sourceIPAddress가 서비스 이름이나 내부 표기인 경우도 구분한다.


In [3]:
events = lesson.load_events(lesson.CLOUDTRAIL_TEXT)
print(json.dumps(result["cloudtrail"], ensure_ascii=False, indent=2))
assert result["cloudtrail"][0]["change_name_candidate"]
assert not result["cloudtrail"][2]["change_name_candidate"]
assert result["cloudtrail"][3]["review_event"]
assert not result["cloudtrail"][3]["change_name_candidate"]


[
  {
    "event_id": "training-01",
    "name": "CreateUser",
    "change_name_candidate": true,
    "review_event": false,
    "access_denied_candidate": false,
    "cli_version_prefix": true,
    "source_kind": "IPv4"
  },
  {
    "event_id": "training-02",
    "name": "DescribeInstances",
    "change_name_candidate": false,
    "review_event": false,
    "access_denied_candidate": true,
    "cli_version_prefix": false,
    "source_kind": "IP로 해석되지 않음"
  },
  {
    "event_id": "training-03",
    "name": "ListUsers",
    "change_name_candidate": false,
    "review_event": false,
    "access_denied_candidate": true,
    "cli_version_prefix": false,
    "source_kind": "IPv6"
  },
  {
    "event_id": "training-04",
    "name": "StopLogging",
    "change_name_candidate": false,
    "review_event": true,
    "access_denied_candidate": false,
    "cli_version_prefix": false,
    "source_kind": "IP로 해석되지 않음"
  }
]


## Checks

정상·유사·경계 입력으로 패턴의 범위를 확인한다. 누락 필드는 불일치로 처리하고 잘못된 자료형은 별도 오류로 구분한다.


In [4]:
assert lesson.ACCESS_DENIED.fullmatch("AccessDeniedButAllowed") is None
assert lesson.CLI_AGENT.match("custom aws-cli/2.0.0") is None
assert lesson.ACCOUNT_ID.fullmatch("１２３４５６７８９０１２") is None
assert lesson.SHA256.fullmatch("g" * 64) is None
assert lesson.find_ssh_failure(lesson.AUTH_LINES[0] + " trailing") is None
assert lesson.classify_cloudtrail({"userAgent": None})["cli_version_prefix"] is False
try:
    lesson.classify_cloudtrail({"userAgent": []})
except TypeError:
    pass
else:
    raise AssertionError("잘못된 자료형을 거부해야 한다")
print("보안 로그 정규식 경계 검사 통과")


보안 로그 정규식 경계 검사 통과


## Next Steps

1. URL과 User-Agent만 변경한 웹로그를 추가한다.
2. CloudTrail 필드 순서를 바꿔 결과를 비교한다.
3. 05-4에서는 이름 있는 그룹으로 필드를 추출하고, 05-5에서는 IP·포트·날짜의 의미를 검증한다.

AWS 구조는 [필드 정의](https://docs.aws.amazon.com/awscloudtrail/latest/userguide/cloudtrail-event-reference-record-contents.html), 웹 형식은 [Apache 문서](https://httpd.apache.org/docs/2.4/logs.html)를 참고한다.
